### Import Dependencise

In [8]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams,Distance,PayloadSchemaType,PointStruct,SparseVectorParams,Document,Prefetch,FusionQuery
from qdrant_client import models
import pandas as pd
from openai import OpenAI
import fastembed
import os

c:\Users\jaysi\Desktop\Desktop\Ai-engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [46]:
import openai

In [9]:
client=OpenAI()

In [8]:
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6333")

### Create Quadrant Collection For Hybrid Collection

In [70]:
quadrant_client=QdrantClient(url="http://localhost:6333/")

In [ ]:
quadrant_client.create_collection(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    vectors_config={
        "text-embedding-model-3-small":VectorParams(size=1536,distance=Distance.COSINE)
    }
    ,
    sparse_vectors_config={
        "bm25":SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

In [11]:
def get_embedding(text,model="text-embedding-3-small"):
    response=openai.embeddings.create(
        input=[text],
        model=model
    )
    return response.data[0].embedding

In [44]:
def get_embedding_batch(text_list,model="text-embedding-3-small",batch_size=100):
    if len(text_list)<=batch_size:
        response=openai.embeddings.create(input=text_list,model=model)
        return [embedding.embedding for embedding in response.data]

    all_embeddings=[]
    counter=1
    for i in range(0,len(text_list),batch_size):
        batch=text_list[i:i+batch_size]
        response=openai.embeddings.create(input=batch,model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        counter+=1
    
    return all_embeddings
        

### Preprocesse Amazon data

In [12]:
df_items=pd.read_json("C:\\Users\\jaysi\\Desktop\\Desktop\\Ai-engineering\\data\\meta_Electronics_2022_23_with_categeory_rating_100_sample_2000.jsonl",lines=True)

In [13]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[],169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...",[],24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN


In [14]:
def preprocessed_decription(row):
    return f"{row['title']}{''.join(row['features'])}"

In [15]:
def extrect_first_large_image(row):
    return row['images'][0].get("large","")

In [16]:
df_items['description']=df_items.apply(preprocessed_decription,axis=1)
df_items['image']=df_items.apply(extrect_first_large_image,axis=1)

In [32]:
df_items


,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51G07yWoOB...
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41bOA5-ogW...
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,USB C Docking Station Dual Monitor for MacBook...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN,https://m.media-amazon.com/images/I/416IzmVKiC...
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[2023 Upgraded] Telescopes for Adults Astronom...,169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41wO4J3TT0...
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...","Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41mwlYqT5p...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Office Products,"WALI Single Monitor Stand, Adjustable Gas Spri...",4.2,210,[Compatibility: The single monitor arm fits mo...,"WALI Single Monitor Stand, Adjustable Gas Spri...",47.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'WALI Single Monitor Stand White (G...,WALI,"[Electronics, Computers & Accessories, Compute...","{'Manufacturer': 'WALI', 'Brand': 'WALI', 'Ite...",B09RTDQ8LZ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31acj+ZylJ...
1996,Industrial & Scientific,"ARESGAME CPU Air Cooler for Intel/AMD, with 5 ...",4.6,1252,[Universal Socket Compatibility: Intel LGA775/...,"ARESGAME CPU Air Cooler for Intel/AMD, with 5 ...",58.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': '4 Heat-pipes CPU Air Cooler-CR1400...,ARESGAME,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'ARESGAME', 'Power Connector Type': ...",B0B6FBGY9H,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41fIjMB-WZ...
1997,All Electronics,BenQ TH585P 1080p Home Entertainment Projector...,4.5,327,[1080P RESOLUTION: 1080p Full HD image quality...,BenQ TH585P 1080p Home Entertainment Projector...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'TH671ST - Low Input Lag', 'url': '...",BenQ,"[Electronics, Video Projectors]",{'Product Dimensions': '18.1 x 13.5 x 7 inches...,B09WF96S4C,NaN,NaN,NaN,https://m.media-amazon.com/images/I/31twaVX106...
1998,Industrial & Scientific,Meboyz 3 Pack 100W USB C to USB C Cable 10ft+6...,4.6,561,[【5A Rapid C

In [33]:
data_to_embed=df_items[["description","image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [37]:
data_to_embed[0]['description']

"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB Type C Fast Charging Cord - Nylon Braided USB C Charger Cable for Galaxy A20/A50/S10/S9/S8+/S8, iPad Pro 2018, Sony XZ, HTC 10, OnePlus 5T, Huawei P9 etc.【Fast Charging Cord】These USB C cables provide up to a 3A charging current to greatly shorten the charging time, meets QC2.0 /3.0 fast charging protocol,Incredibly charge your phone from 0 to 80% in 50 minute. 480Mbps (40-60M/s) ultra fast data transmission, which leads to a faster data sync.(Note:Cables support fast charging,but require a USB-A QC3.0/QC2.0/AFC charger)【Universal Compatibility】The USB C Charger Cable is compatible with Samsung Galaxy S20 / S10 / S9 / S8+ / S8 / A02s / A03s,A12 A20 A21 A22 A23 A31 A32 A33 A41 A42 A50 A52 A52s 5G A71 A72 A73,M11 M21 M31 M51,M12 M22 M32 M52,iPad Pro 2018 / 2020,Sony Xperia XZ/X Compact/L1 / XZs / XA1 / X Premium, HTC 10 LG G5 G6,OnePlus 5T / 6T, Lumia 950 / 950XL,Huawei P9 P9 Plus P10 P10 Plus Honor Mate 9 Mate 9 pro Mate 10 pro Mate 10 lite 

In [38]:
text_to_embed=[data["description"] for data in data_to_embed]

In [59]:
len(text_to_embed)

2000

In [ ]:
embeddings = get_embedding_batch(text_to_embed)


In [60]:
len(embedding)

1536

In [64]:
embedding

[0.037017822265625,
 -0.005466461181640625,
 -0.02996826171875,
 0.0200653076171875,
 0.0010128021240234375,
 -0.059295654296875,
 -0.005069732666015625,
 0.01007843017578125,
 0.03680419921875,
 -0.004947662353515625,
 -0.01247406005859375,
 0.023406982421875,
 -0.06396484375,
 -0.0243072509765625,
 0.006504058837890625,
 0.06024169921875,
 -0.0173492431640625,
 0.001476287841796875,
 -0.024444580078125,
 -0.02276611328125,
 -0.00363922119140625,
 0.06976318359375,
 0.0258331298828125,
 -0.03106689453125,
 0.005474090576171875,
 0.002216339111328125,
 -0.019866943359375,
 0.00975799560546875,
 0.0175628662109375,
 -0.018035888671875,
 0.00496673583984375,
 -0.030120849609375,
 0.03265380859375,
 -0.0469970703125,
 0.0210418701171875,
 0.038970947265625,
 0.00441741943359375,
 -0.0078277587890625,
 0.032196044921875,
 0.0139312744140625,
 0.040802001953125,
 0.0267486572265625,
 0.05718994140625,
 -0.07501220703125,
 0.0191192626953125,
 0.05322265625,
 -0.034149169921875,
 0.012336730

In [63]:
import numpy as np
np.array(embeddings[0]).shape


(1536,)

In [62]:
len(embeddings[0])


1536

In [58]:
pointstructs=[]
i=1
for embedding,data in zip(embeddings,data_to_embed):
    if i==1:print(embedding,data)
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-model-3-small":embedding,
                "bm25":Document(
                    text=data["description"],
                    model="qdrant/bm25"
                )
            },
            payload=data
        )
    )
    i+=1

[0.01305389404296875, -0.030670166015625, -0.01507568359375, 0.00897216796875, -0.046783447265625, -0.0269775390625, -0.049163818359375, 0.02447509765625, 0.061279296875, -0.033905029296875, -0.0159759521484375, -0.0028553009033203125, -0.06103515625, -0.0266265869140625, 0.03350830078125, 0.0217132568359375, -0.051116943359375, -0.0274810791015625, -0.0243377685546875, 0.0068817138671875, 0.006885528564453125, 0.022674560546875, 0.0208587646484375, 0.0187225341796875, -0.026824951171875, -0.032012939453125, -0.01806640625, 0.029083251953125, -0.0130462646484375, -0.0087127685546875, -0.044342041015625, -0.0226287841796875, 0.003875732421875, -0.0204620361328125, -0.048187255859375, 0.054962158203125, -0.01042938232421875, 0.0060577392578125, 0.038116455078125, -0.0089111328125, 0.0158538818359375, 0.0040740966796875, 0.0494384765625, -0.0297088623046875, 0.01210784912109375, -0.006237030029296875, -0.02423095703125, 0.0308837890625, 0.0144500732421875, 0.0213470458984375, -0.021957397

In [ ]:
pointstructs

In [10]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[0:500],
    wait=True
)

NameError: name 'quadrant_client' is not defined

In [24]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[500:1000],
    wait=True
)

UpdateResult(operation_id=2, status=<UpdateStatus.COMPLETED: 'completed'>)

In [25]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[1000:1500],
    wait=True
)

UpdateResult(operation_id=3, status=<UpdateStatus.COMPLETED: 'completed'>)

In [34]:
quadrant_client.upsert(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    points=pointstructs[1500:2000],
    wait=True
)

UpdateResult(operation_id=5, status=<UpdateStatus.COMPLETED: 'completed'>)

### Hybrid Retrival

In [11]:
def get_embedding(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding

In [14]:
qdrant_client=QdrantClient(url="http://localhost:6333/")
def get_only_dense(query):
    query_embedding = get_embedding(query)

    dense_result = quadrant_client.query_points(
        collection_name="Amazon-items-collection-02-hybrid-serach",
        query=query_embedding,
        using="text-embedding-model-3-small",
        limit=10
    )

    for point in dense_result.points:
        print(point.score, point.payload["description"])


In [84]:
get_only_dense("can i get some tablet")

0.425358 ALZHIJ Portable Tablet Holder Stand - Aluminum Alloy Desktop Stand Adjustable Angle&Height Tablet Stand Holder Creative Tablet Stand Compatible with Various Tablet UP to 12IN Wide (Grey)😃MADE OF HIGH-QUALITY MATERIALS:The high-quality aluminum alloy material achieves the perfect performance of the tablet holder for bed, making the tablet wear-resistant and scratch-resistant, and has excellent corrosion resistance. The superb anodizing process increases the color and luster of the table stand and makes it richer in texture.😃EXQUISITE APPEARANCE DESIGN:The personalized smiley face design can not only be a better decoration for this tablet stand for desk, but also provide a long-lasting heat dissipation effect for the tablet during use, making your tablet run more efficiently. The tablet stand for bed is designed with multiple non-slip silicone pads to protect the tablet from damage at all times.😃180°FREELY ADJUSTABLE ANGLE:The tablet floor stand adopts a stable dual-axis mechani

In [85]:
def get_only_Bm25(query):
    query_embedding = get_embedding(query)
    
    bm25_result = qdrant_client.query_points(
    collection_name="Amazon-items-collection-02-hybrid-serach",
    query=Document(
        text="can i get some tablet",
        model="qdrant/bm25"
    ),
    using="bm25",
    limit=10
   )
    for point in bm25_result.points:
      print(point.score, point.payload["description"])



In [86]:
get_only_Bm25("can i get some tablet")

5.8474283 Android Tablet 10.1 Inch, Tablet with Keyboard, 2 in 1 Tablets 64GB Memory 4G RAM 128GB Expandable, 5G WiFi Tablet, Android 11, 13MP Camera, Large Battery, Tablet with Case, Keyboard, Mouse, StylusAndroid 11.0 Standard OS Google Tablet: 2022 Newest android tablet runs the latest android 11.0 OS, which is easier when set up and operate. Google base can better protect your privacy and compatible with more apps in pre-installed google play store. Get yourself a wonderful tablet and all your favourite apps on it.WiFi tablet Support Dual band 2.4+5Ghz: 5G WiFi tablet support not only 5Ghz Wifi but also 2.4Ghz wifi meet anyone needs. No restrictions on any usage environment, whether you live in a hotel/apartment with only 5Ghz WiFi, or in your home with 2.4Ghz wifi, it can be easily connected, provides the best speed when using this tablet.Tablet 64GB Storage 128GB Micro SD Expandable: 2 in 1 android 11 tablet has a 1.6 GHz high-efficiency Quad-core processor, 4GB RAM+64GB ROM, fas

In [19]:
def get_both_denceandBM25(query,k=20):
    query_embedding = get_embedding(query)
    result = qdrant_client.query_points(
        collection_name="Amazon-items-collection-02-hybrid-serach",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-model-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )

        ],
        query=FusionQuery(fusion="rrf"),
        limit=20
    )

    return result

In [20]:
result=get_both_denceandBM25("can i get some Tablet")

In [25]:
for point in result.points:
    print(point.payload['description'])

Android Tablet 10.1 Inch, Tablet with Keyboard, 2 in 1 Tablets 64GB Memory 4G RAM 128GB Expandable, 5G WiFi Tablet, Android 11, 13MP Camera, Large Battery, Tablet with Case, Keyboard, Mouse, StylusAndroid 11.0 Standard OS Google Tablet: 2022 Newest android tablet runs the latest android 11.0 OS, which is easier when set up and operate. Google base can better protect your privacy and compatible with more apps in pre-installed google play store. Get yourself a wonderful tablet and all your favourite apps on it.WiFi tablet Support Dual band 2.4+5Ghz: 5G WiFi tablet support not only 5Ghz Wifi but also 2.4Ghz wifi meet anyone needs. No restrictions on any usage environment, whether you live in a hotel/apartment with only 5Ghz WiFi, or in your home with 2.4Ghz wifi, it can be easily connected, provides the best speed when using this tablet.Tablet 64GB Storage 128GB Micro SD Expandable: 2 in 1 android 11 tablet has a 1.6 GHz high-efficiency Quad-core processor, 4GB RAM+64GB ROM, fast and resp

In [ ]:
def reteriver_data(query, quadrant_client, k):
    
    query_embedding = get_embedding(query)
    result = quadrant_client.query_points(
        collection_name="Amazon-items-collection-02-hybrid-serach",
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-model-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )

        ],
        query=FusionQuery(fusion="rrf"),
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_rating = []
    retrieved_image_urls = []
    retrieved_prices = []

    for point in result.points:
        payload = point.payload or {}
        retrieved_context_ids.append(payload.get("parent_asin"))
        retrieved_context.append(payload.get("description", ""))
        retrieved_context_rating.append(payload.get("average_rating"))
        similarity_scores.append(point.score)
        retrieved_image_urls.append(payload.get("image", ""))
        retrieved_prices.append(payload.get("price"))

    return (
        retrieved_context,
        retrieved_context_ids,
        retrieved_context_rating,
        similarity_scores,
        retrieved_image_urls,
        retrieved_prices
    )


In [68]:
result=reteriver_data("can i get some ipad",quadrant_client,k=20)

In [45]:
result

(['Case for iPad Pro 12.9 2022/2021/2020/2018 Gen 6th/5th/4th/3rd,2 in 1 Detachable Magnetic Case,Anti-Fingerprint Frosted Case and Washable Leather Protective Cover Support Pen Charger(Pink)✅【Protective Case for iPad Pro 12.9】❗❗No Pen included!❗❗The protective cover is designed for iPad Pro 12.9" 6th Gen 2022 (Model number: A2764/A2436/A2437/A2766),5th Gen 2021 (Model number: A2378/A2379/A2461/A2462),4th Gen 2020 (A2229/A2069/A2232/A2233),3rd Gen 2018 (A1876/A1895/A1983/A2014).♐It\'s Not compatible with any other devices. Please check the model number of your iPad back bottom.✅【2 in 1 Detachable & Magnetic Cover】The detachable design is more convenient to carry and use. When it\'s used as a protective case, you have a video mode or a writing mode. When it detached, you get the phone holder.✅【Washable & Stain-resistant Case】The premium leather and frosted case are both washable. You can easily get a new look on your case with a single tissue, even if it\'s stained with coffee, oil, or 